# NB02 — Association Analysis
**Pipeline:** Generalised Metabolite–Metagenomics Correlation Study  
**Inputs:** `preprocessed_data.pkl`  
**Outputs:** `analysis_results.pkl`, CSV tables, figures

| Step | Analysis |
|---|---|
| 1–3 | Load data, feature selection (top-variance) |
| 4 | Differential abundance — 3 comparisons (Healthy vs Early, Healthy vs Advanced, Early vs Advanced) |
| 5 | Volcano plots — metabolites + species DA heatmap |
| 6 | Global Spearman correlation matrix (vectorised) |
| 7 | Partial correlations (confounder-adjusted OLS residuals) |
| 8 | Stage-stratified Spearman correlations |
| 9 | Species co-abundance matrix |
| 10 | Hub species / metabolite identification |
| 11 | Kim_adenomas_2020 replication |
| 12 | Save results |


## 1 · Imports

In [1]:
import sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter

sys.path.insert(0, str(Path(".").resolve()))

from utils import (
    RESULTS_DIR, INTER_DIR, FIG_DIR, TABLE_DIR,
    DATASET_PRIMARY, DATASET_SECONDARY,
    CRC_STAGE_ORDER, THREE_GROUP_ORDER,
    FDR_THRESHOLD, MIN_CORR,
    MAX_SPECIES_NB02, MAX_MTB_NB02,
    PALETTE_STAGE6, PALETTE_3GROUP,
    load_pickle, save_pickle, savefig,
    differential_abundance,
    spearman_correlation_matrix,
    partial_corr_residuals,
    species_coabundance_matrix,
    annotate_pathway, pathway_kegg_ids,
)
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore")
%matplotlib inline
%config InlineBackend.figure_format = "retina"
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100

for d in [FIG_DIR/"differential", FIG_DIR/"correlation", TABLE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Imports OK")
print(f"  Feature caps: {MAX_SPECIES_NB02} species  |  {MAX_MTB_NB02} metabolites")


Imports OK
  Feature caps: 500 species  |  150 metabolites


---
## 2 · Load Preprocessed Data

In [2]:
pkl = load_pickle(INTER_DIR / "preprocessed_data.pkl")

harmonized_meta  = pkl["harmonized_meta"]
species_clr_all  = pkl["species_clr"]
mtb_log10_all    = pkl["mtb_log10"]
name_maps        = pkl["name_maps"]

spe_full = species_clr_all[DATASET_PRIMARY].copy()
mtb_full = mtb_log10_all[DATASET_PRIMARY].copy()
meta_yk  = harmonized_meta[DATASET_PRIMARY].loc[spe_full.index]
nm_y     = name_maps.get(DATASET_PRIMARY, {})

print(f"YACHIDA: {spe_full.shape[1]} species  |  {mtb_full.shape[1]} metabolites  |  {len(meta_yk)} samples")
print("Stage distribution:")
print(meta_yk["Stage.3Group"].value_counts().reindex(THREE_GROUP_ORDER).fillna(0).astype(int))


Loaded: C:\Users\andna\Documents\KI\Results\intermediate\preprocessed_data.pkl
YACHIDA: 4392 species  |  249 metabolites  |  347 samples
Stage distribution:
Stage.3Group
Healthy         127
Early_CRC        57
Advanced_CRC    163
Name: count, dtype: int64


---
## 3 · Feature Selection (top-variance)
Reduces FDR multiple-testing burden without pathway pre-filtering.

In [3]:
def top_variance_select(df: pd.DataFrame, n: int) -> pd.DataFrame:
    if df.shape[1] <= n:
        return df
    top_cols = df.var(axis=0).nlargest(n).index
    return df[top_cols]

spe = top_variance_select(spe_full, MAX_SPECIES_NB02)
mtb = top_variance_select(mtb_full, MAX_MTB_NB02)

print(f"After feature selection:")
print(f"  Species    : {spe_full.shape[1]:>4} -> {spe.shape[1]}")
print(f"  Metabolites: {mtb_full.shape[1]:>5} -> {mtb.shape[1]}")


After feature selection:
  Species    : 4392 -> 500
  Metabolites:   249 -> 150


---
## 4 · Differential Abundance — 3 Comparisons
Mann-Whitney U + BH correction.  
Comparisons: Healthy vs Early_CRC · Healthy vs Advanced_CRC · Early_CRC vs Advanced_CRC

In [4]:
COMPARISONS = [
    ("Healthy", "Early_CRC"),
    ("Healthy", "Advanced_CRC"),
    ("Early_CRC", "Advanced_CRC"),
]

da_mtb = {}
da_spe = {}

for g1, g2 in COMPARISONS:
    label = f"{g1}_vs_{g2}"
    da_mtb[label] = differential_abundance(mtb, meta_yk, "Stage.3Group", g1, g2)
    da_spe[label] = differential_abundance(spe, meta_yk, "Stage.3Group", g1, g2)
    n_ms = (da_mtb[label]["qval"] < FDR_THRESHOLD).sum() if not da_mtb[label].empty else 0
    n_ss = (da_spe[label]["qval"] < FDR_THRESHOLD).sum() if not da_spe[label].empty else 0
    print(f"  {label:<38}  metabolites q<0.05: {n_ms:>4}  |  species q<0.05: {n_ss:>4}")

# Save DA tables
for comp, da in da_mtb.items():
    if not da.empty:
        da["metabolite_name"] = da["feature"].map(nm_y).fillna(da["feature"])
        da["pathway"] = da["feature"].apply(annotate_pathway)
        da.to_csv(TABLE_DIR / f"da_metabolites_{comp}.csv", index=False)
for comp, da in da_spe.items():
    if not da.empty:
        da.to_csv(TABLE_DIR / f"da_species_{comp}.csv", index=False)


  Healthy_vs_Early_CRC                    metabolites q<0.05:    3  |  species q<0.05:    0
  Healthy_vs_Advanced_CRC                 metabolites q<0.05:    1  |  species q<0.05:    2
  Early_CRC_vs_Advanced_CRC               metabolites q<0.05:    6  |  species q<0.05:    3


---
## 5 · Volcano Plots & Species DA Heatmap

In [5]:
# Volcano plots — metabolites, Healthy vs Early and Healthy vs Advanced
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, comp in zip(axes, ["Healthy_vs_Early_CRC", "Healthy_vs_Advanced_CRC"]):
    da = da_mtb.get(comp, pd.DataFrame())
    if da.empty:
        ax.set_title(f"{comp} (no data)")
        continue
    sig  = da["qval"] < FDR_THRESHOLD
    up   = sig & (da["log2FC"] > 0)
    down = sig & (da["log2FC"] < 0)
    ns   = ~sig

    ax.scatter(da.loc[ns,   "log2FC"], -np.log10(da.loc[ns,   "pval"] + 1e-300),
               c="#BDBDBD", s=12, alpha=0.5, label="NS")
    ax.scatter(da.loc[up,   "log2FC"], -np.log10(da.loc[up,   "pval"] + 1e-300),
               c="#F44336", s=18, alpha=0.8, label=f"Up in CRC ({up.sum()})")
    ax.scatter(da.loc[down, "log2FC"], -np.log10(da.loc[down, "pval"] + 1e-300),
               c="#2196F3", s=18, alpha=0.8, label=f"Down in CRC ({down.sum()})")

    # Label top 8 by absolute effect size
    top8 = da[sig].nlargest(min(8, sig.sum()), "effect_size") if sig.any() else pd.DataFrame()
    for _, row in top8.iterrows():
        name = nm_y.get(row["feature"], row["feature"])[:22]
        ax.text(row["log2FC"], -np.log10(row["pval"] + 1e-300) + 0.15,
                name, fontsize=6, ha="center")

    ax.axhline(-np.log10(0.05), color="grey", ls="--", lw=0.8)
    ax.axvline(0, color="grey", ls="-", lw=0.5)
    ax.set_xlabel("log2 Fold Change (CRC / Healthy)")
    ax.set_ylabel("-log10(p-value)")
    ax.set_title(comp.replace("_", " "))
    ax.legend(fontsize=8)

plt.suptitle("Differential Metabolites — YACHIDA", fontweight="bold")
plt.tight_layout()
savefig(fig, "differential", "nb02_volcano_metabolites.png")


Saved figure: C:\Users\andna\Documents\KI\Results\figures\differential\nb02_volcano_metabolites.png


In [6]:
# Species DA heatmap — top 20 DA species (Healthy vs Advanced_CRC)
comp  = "Healthy_vs_Advanced_CRC"
da_s  = da_spe.get(comp, pd.DataFrame())

if not da_s.empty:
    sig_spe = da_s[da_s["qval"] < FDR_THRESHOLD]["feature"].tolist()
    top_spe = sig_spe[:20] if len(sig_spe) >= 2 else da_s.head(20)["feature"].tolist()

    groups  = meta_yk[meta_yk["Stage.3Group"].isin(["Healthy", "Advanced_CRC"])]
    hm_data = spe.loc[groups.index, [c for c in top_spe if c in spe.columns]]
    hm_data = (hm_data - hm_data.mean()) / (hm_data.std() + 1e-9)

    row_colors = groups["Stage.3Group"].map(PALETTE_3GROUP)
    g = sns.clustermap(
        hm_data.T,
        col_colors=row_colors.values,
        cmap="RdBu_r", center=0, vmin=-3, vmax=3,
        yticklabels=True, xticklabels=False,
        figsize=(12, 8), dendrogram_ratio=(0.05, 0.15)
    )
    g.fig.suptitle("Top 20 DA Species (Healthy vs Advanced_CRC) — Z-scored CLR",
                   fontweight="bold", y=1.01)
    g.savefig(FIG_DIR / "differential" / "nb02_species_da_heatmap.png",
              dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Species DA heatmap saved ({len(top_spe)} species)")


Species DA heatmap saved (2 species)


---
## 6 · Global Spearman Correlation (all species × all metabolites)
Vectorised rank-matrix multiplication. |ρ| ≥ 0.10 pre-filter before BH correction.

> **Statistical note (S1):** Spearman rank-correlation is computed on CLR-transformed
> species data. Rank statistics are invariant to monotone transforms, so the CLR does
> not invalidate Spearman ρ. However, CLR transformation does not resolve *compositional
> spurious correlations* arising from the simplex constraint. Partial correlation
> adjustment for confounders (Section 7) partially addresses this, but results should
> be interpreted as exploratory. Note in Methods: "Species–metabolite correlations were
> computed on CLR-transformed species data; compositional bias was partially addressed
> via partial correlation adjustment for Age, BMI, and Sex. Residual compositional
> spurious correlations cannot be excluded and results should be treated as hypothesis-
> generating."

In [7]:
corr_all = spearman_correlation_matrix(spe, mtb, fdr=FDR_THRESHOLD, min_r=MIN_CORR)

print(f"Significant species-metabolite pairs: {len(corr_all)}")
if not corr_all.empty:
    corr_all["pathway"]        = corr_all["metabolite"].apply(annotate_pathway)
    corr_all["metabolite_name"] = corr_all["metabolite"].map(nm_y).fillna(corr_all["metabolite"])
    corr_all.to_csv(TABLE_DIR / "spearman_correlations_all.csv", index=False)
    print("\nTop 10 associations:")
    print(corr_all[["species","metabolite_name","rho","qval","pathway"]].head(10).to_string(index=False))
    print("\nPathway breakdown of significant pairs:")
    print(corr_all["pathway"].value_counts().to_string())


Significant species-metabolite pairs: 14656

Top 10 associations:
                       species metabolite_name       rho         qval   pathway
    Acetatifactor intestinalis          C02678  0.505035 1.275585e-19     Other
           UBA5905 sp900763035          C02714  0.495483 3.456506e-19     Other
               ER4 sp000765235          C05629  0.494758 3.456506e-19     Other
     Acetatifactor sp003447295          C02678  0.495755 3.456506e-19     Other
           UBA5905 sp900763035          C00134  0.486789 1.661596e-18 Polyamine
               ER4 sp000765235          C02678  0.475280 1.701465e-17     Other
           UBA5905 sp900763035          C08277 -0.453589 1.283920e-15     Other
               ER4 sp900552015          C05629  0.452491 1.397374e-15     Other
              Alistipes shahii          C03283  0.445100 5.291012e-15     Other
Faecalibacterium prausnitzii_C          C05332 -0.439333 1.439577e-14     Other

Pathway breakdown of significant pairs:
pathway
Other

In [8]:
# Heatmap: top-50 associations
if len(corr_all) > 0:
    top_pairs = corr_all.head(50)
    hm_spe = top_pairs["species"].unique().tolist()
    hm_mtb = top_pairs["metabolite"].unique().tolist()

    rho_mat = (
        corr_all[
            corr_all["species"].isin(hm_spe) &
            corr_all["metabolite"].isin(hm_mtb)
        ].pivot_table(index="species", columns="metabolite", values="rho", fill_value=0)
    )

    if not rho_mat.empty:
        new_cols = [nm_y.get(c, c)[:25] for c in rho_mat.columns]
        rho_mat.columns = new_cols
        g = sns.clustermap(
            rho_mat, cmap="RdBu_r", center=0, vmin=-0.6, vmax=0.6,
            yticklabels=True, xticklabels=True,
            figsize=(max(8, len(hm_mtb)*0.45), max(6, len(hm_spe)*0.22))
        )
        g.ax_heatmap.set_xticklabels(
            g.ax_heatmap.get_xticklabels(), fontsize=6, rotation=45, ha="right")
        g.fig.suptitle("Top 50 Species-Metabolite Associations (Spearman rho)",
                       y=1.01, fontweight="bold")
        g.savefig(FIG_DIR / "correlation" / "nb02_top50_correlation_heatmap.png",
                  dpi=150, bbox_inches="tight")
        plt.close()
        print("Correlation heatmap saved.")


Correlation heatmap saved.


---
## 7 · Partial Correlations (confounder adjustment)
OLS residuals approach: regress species and metabolite separately on confounders,
then compute Spearman rho on residuals. BH correction applied across all pairs.

In [9]:
conf_candidates = ["Age", "BMI", "Sex", "Gender"]
conf_available  = [
    c for c in conf_candidates
    if c in meta_yk.columns and meta_yk[c].notna().mean() > 0.3
]
print(f"Confounders used for partial correlation: {conf_available}")

partial_results = []

if conf_available and not corr_all.empty:
    cov_df = meta_yk[conf_available].copy()
    for cat_col in ["Sex", "Gender"]:
        if cat_col in cov_df.columns:
            cov_df[cat_col] = pd.factorize(cov_df[cat_col])[0].astype(float)
    cov_df = cov_df.apply(pd.to_numeric, errors="coerce")

    # NB02-1: test all pairs with |rho| >= MIN_CORR, not just top-200.
    # Selecting only top-200 before confounder adjustment is a selection bias —
    # pairs that become significant only after adjustment are never tested.
    # A performance cap of 2000 is applied if corr_all is very large; document in Methods.
    N_PARTIAL_CAP = 2000
    if len(corr_all) > N_PARTIAL_CAP:
        test_pairs = corr_all.head(N_PARTIAL_CAP)
        print(f"Performance cap: testing top {N_PARTIAL_CAP} of {len(corr_all)} "
              f"pairs for partial correlations. Note this selection in Methods.")
    else:
        test_pairs = corr_all

    for _, row in test_pairs.iterrows():
        spe_name = row["species"]
        mtb_name = row["metabolite"]
        if spe_name not in spe.columns or mtb_name not in mtb.columns:
            continue
        common_idx = (
            spe.index.intersection(mtb.index)
            .intersection(cov_df.dropna().index)
        )
        if len(common_idx) < 20:
            continue
        x_vec = spe.loc[common_idx, spe_name].values.astype(float)
        y_vec = mtb.loc[common_idx, mtb_name].values.astype(float)
        cov   = cov_df.loc[common_idx].values.astype(float)
        rho_p, pval_p = partial_corr_residuals(x_vec, y_vec, cov)
        partial_results.append({
            "species":      spe_name,
            "metabolite":   mtb_name,
            "rho":          row["rho"],
            "qval":         row["qval"],
            "rho_partial":  rho_p,
            "pval_partial": pval_p,
            "pathway":      row.get("pathway", annotate_pathway(mtb_name)),
        })

    corr_partial = pd.DataFrame(partial_results)
    if not corr_partial.empty:
        valid_p = corr_partial["pval_partial"].notna()
        if valid_p.sum() > 1:
            _, qvals_p, _, _ = multipletests(
                corr_partial.loc[valid_p, "pval_partial"], method="fdr_bh")
            corr_partial.loc[valid_p, "qval_partial"] = qvals_p
        corr_partial["survives_adjustment"] = corr_partial["qval_partial"] < FDR_THRESHOLD
        corr_partial_sig = corr_partial[corr_partial["survives_adjustment"]].copy()
        print(f"Pairs tested: {len(corr_partial)}  |  surviving adjustment: {len(corr_partial_sig)}")
        corr_partial.to_csv(TABLE_DIR / "spearman_correlations_partial.csv", index=False)
    else:
        corr_partial     = pd.DataFrame()
        corr_partial_sig = pd.DataFrame()
else:
    print("No confounders available or no significant pairs — skipping partial correlations.")
    corr_partial     = pd.DataFrame()
    corr_partial_sig = pd.DataFrame()


Confounders used for partial correlation: ['Age', 'BMI', 'Gender']
Performance cap: testing top 2000 of 14656 pairs for partial correlations. Note this selection in Methods.
Pairs tested: 2000  |  surviving adjustment: 1997


---
## 8 · Stage-Stratified Correlations
Separate Spearman analyses within Healthy / Early_CRC / Advanced_CRC.
Pairs present in Healthy but absent in Advanced_CRC may reflect disrupted commensal function.

In [10]:
strat_corr = {}

for group in THREE_GROUP_ORDER:
    idx = meta_yk[meta_yk["Stage.3Group"] == group].index
    idx = idx.intersection(spe.index).intersection(mtb.index)
    if len(idx) < 20:
        print(f"  {group:<15}  n={len(idx)} — skipped (< 20 samples)")
        continue
    sp_g   = spe.loc[idx]
    mt_g   = mtb.loc[idx]
    corr_g = spearman_correlation_matrix(sp_g, mt_g, fdr=FDR_THRESHOLD, min_r=MIN_CORR)
    strat_corr[group] = corr_g
    print(f"  {group:<15}  n={len(idx):>4}  significant pairs: {len(corr_g)}")

# Pair overlap summary
if len(strat_corr) >= 2:
    pair_sets = {g: set(zip(df["species"], df["metabolite"]))
                 for g, df in strat_corr.items()}
    groups = list(pair_sets.keys())
    print("\nOverlap of significant pairs:")
    for i, g1 in enumerate(groups):
        for g2 in groups[i+1:]:
            shared = pair_sets[g1] & pair_sets[g2]
            print(f"  {g1} ∩ {g2}: {len(shared)} pairs")
    for g in groups:
        others = set().union(*[v for k, v in pair_sets.items() if k != g])
        unique = pair_sets[g] - others
        print(f"  Unique to {g}: {len(unique)} pairs")


  Healthy          n= 127  significant pairs: 2689
  Early_CRC        n=  57  significant pairs: 310
  Advanced_CRC     n= 163  significant pairs: 2939

Overlap of significant pairs:
  Healthy ∩ Early_CRC: 133 pairs
  Healthy ∩ Advanced_CRC: 737 pairs
  Early_CRC ∩ Advanced_CRC: 139 pairs
  Unique to Healthy: 1914 pairs
  Unique to Early_CRC: 133 pairs
  Unique to Advanced_CRC: 2158 pairs


---
## 9 · Species Co-Abundance Correlations

In [11]:
coabund = species_coabundance_matrix(spe, fdr=FDR_THRESHOLD, min_r=0.20)
print(f"Significant species co-abundance pairs (|rho|>=0.20, q<0.05): {len(coabund)}")

if not coabund.empty:
    coabund.to_csv(TABLE_DIR / "species_coabundance.csv", index=False)

    degree = Counter(coabund["species_a"].tolist() + coabund["species_b"].tolist())
    top_spe_ca = [s for s, _ in degree.most_common(30) if s in spe.columns]

    if len(top_spe_ca) >= 3:
        rho_sym = pd.DataFrame(
            np.eye(len(top_spe_ca)),
            index=top_spe_ca, columns=top_spe_ca
        )
        sub = coabund[
            coabund["species_a"].isin(top_spe_ca) &
            coabund["species_b"].isin(top_spe_ca)
        ]
        for _, row in sub.iterrows():
            rho_sym.loc[row["species_a"], row["species_b"]] = row["rho"]
            rho_sym.loc[row["species_b"], row["species_a"]] = row["rho"]

        g = sns.clustermap(
            rho_sym, cmap="RdBu_r", center=0, vmin=-1, vmax=1,
            figsize=(10, 10), yticklabels=True, xticklabels=True
        )
        g.fig.suptitle("Species Co-Abundance (|rho|>=0.20, top 30 by degree)",
                       y=1.01, fontweight="bold")
        g.savefig(FIG_DIR / "correlation" / "nb02_species_coabundance_heatmap.png",
                  dpi=150, bbox_inches="tight")
        plt.close()
        print("Co-abundance heatmap saved.")


Significant species co-abundance pairs (|rho|>=0.20, q<0.05): 7671
Co-abundance heatmap saved.


---
## 10 · Hub Species / Metabolite Identification
Hub: connected to >= 3 nodes in the other modality after BH correction.

In [12]:
hub_spe = pd.Series(dtype=int)
hub_mtb = pd.Series(dtype=int)

if not corr_all.empty:
    spe_degree = corr_all.groupby("species")["metabolite"].nunique().rename("metabolite_degree")
    mtb_degree = corr_all.groupby("metabolite")["species"].nunique().rename("species_degree")

    hub_spe = spe_degree[spe_degree >= 3].sort_values(ascending=False)
    hub_mtb = mtb_degree[mtb_degree >= 3].sort_values(ascending=False)

    print(f"Hub species (connected to >= 3 metabolites): {len(hub_spe)}")
    print(hub_spe.head(10).to_string())

    print(f"\nHub metabolites (connected to >= 3 species): {len(hub_mtb)}")
    hub_mtb_df = hub_mtb.head(10).reset_index()
    hub_mtb_df["name"]    = hub_mtb_df["metabolite"].map(nm_y)
    hub_mtb_df["pathway"] = hub_mtb_df["metabolite"].apply(annotate_pathway)
    print(hub_mtb_df.to_string(index=False))

    hub_spe.to_csv(TABLE_DIR / "hub_species.csv")
    hub_mtb_df.to_csv(TABLE_DIR / "hub_metabolites.csv", index=False)


Hub species (connected to >= 3 metabolites): 493
species
ER4 sp000765235               92
Alistipes shahii              91
Acetatifactor intestinalis    86
Acetatifactor sp003447295     86
UBA5905 sp900763035           85
ER4 sp900552015               84
CAG-170 sp000432135           83
Faecousia sp003525905         82
Alistipes putredinis          82
CAG-103 sp900543625           78

Hub metabolites (connected to >= 3 species): 150
               metabolite  species_degree   name   pathway
        C00134_Putrescine             253 C00134 Polyamine
          C08277_Sebacate             239 C08277     Other
   C00431_5-Aminovalerate             227 C00431     Other
        C01672_Cadaverine             227 C01672 Polyamine
    C02678_Dodecanedioate             223 C02678     Other
          C02656_Pimelate             220 C02656     Other
C02714_N-Acetylputrescine             217 C02714     Other
           C00695_Cholate             209 C00695 Bile Acid
           C06104_Adipate       

---
## 11 · Replication in Kim_adenomas_2020
Kim adenoma vs control validates early-stage YACHIDA findings.

In [13]:
kim_key  = "Kim_adenomas_2020"
corr_kim = pd.DataFrame()
da_kim   = pd.DataFrame()

if kim_key in species_clr_all and kim_key in mtb_log10_all:
    spe_k  = top_variance_select(species_clr_all[kim_key], MAX_SPECIES_NB02)
    mtb_k  = top_variance_select(mtb_log10_all[kim_key],   MAX_MTB_NB02)
    meta_k = pkl["harmonized_meta"][kim_key].loc[spe_k.index]

    da_kim = differential_abundance(mtb_k, meta_k, "Stage.3Group", "Healthy", "Early_CRC")
    n_sig_kim = (da_kim["qval"] < FDR_THRESHOLD).sum() if not da_kim.empty else 0
    print(f"Kim DA (control vs adenoma): {n_sig_kim} significant metabolites at q<0.05")

    corr_kim = spearman_correlation_matrix(spe_k, mtb_k, fdr=FDR_THRESHOLD, min_r=MIN_CORR)
    print(f"Kim significant pairs: {len(corr_kim)}")

    if not corr_all.empty and not corr_kim.empty:
        yk_pairs  = set(zip(corr_all["species"], corr_all["metabolite"]))
        kim_pairs = set(zip(corr_kim["species"], corr_kim["metabolite"]))
        overlap   = yk_pairs & kim_pairs
        print(f"\nYACHIDA-Kim overlap: {len(overlap)} shared species-metabolite pairs")
        print(f"  (YACHIDA n={len(yk_pairs)}, Kim n={len(kim_pairs)})")
        print(f"  Replication rate: {len(overlap)/len(yk_pairs)*100:.1f}% of YACHIDA pairs found in Kim")
else:
    print(f"Kim_adenomas_2020 not loaded — check NB01 data loading.")


Kim DA (control vs adenoma): 0 significant metabolites at q<0.05
Kim significant pairs: 6368

YACHIDA-Kim overlap: 0 shared species-metabolite pairs
  (YACHIDA n=14656, Kim n=6368)
  Replication rate: 0.0% of YACHIDA pairs found in Kim


---
## 12 · Save Results

In [14]:
# Use local variable checks (not dir()) to safely reference optional variables
analysis_results = {
    "spe":              spe,
    "mtb":              mtb,
    "meta_yk":          meta_yk,
    "da_mtb":           da_mtb,
    "da_spe":           da_spe,
    "corr_all":         corr_all,
    "corr_partial":     corr_partial,
    "corr_partial_sig": corr_partial_sig,
    "strat_corr":       strat_corr,
    "coabund":          coabund,
    "corr_kim":         corr_kim,
    "da_kim":           da_kim,
    "hub_spe":          hub_spe,
    "hub_mtb":          hub_mtb,
    "nm_y":             nm_y,
}

save_pickle(analysis_results, INTER_DIR / "analysis_results.pkl")

print("\n-- NB02 complete ---------------------------------------------------")
print(f"  Significant pairs (global)           : {len(corr_all)}")
print(f"  Surviving confounder adjustment      : {len(corr_partial_sig)}")
print(f"  Hub species (>= 3 metabolite edges)  : {len(hub_spe)}")
print(f"  Hub metabolites (>= 3 species edges) : {len(hub_mtb)}")
print(f"  Significant co-abundance pairs       : {len(coabund)}")
print("  -> Run NB03 (03_ml_benchmarking.ipynb) next.")


Saved: C:\Users\andna\Documents\KI\Results\intermediate\analysis_results.pkl

-- NB02 complete ---------------------------------------------------
  Significant pairs (global)           : 14656
  Surviving confounder adjustment      : 1997
  Hub species (>= 3 metabolite edges)  : 493
  Hub metabolites (>= 3 species edges) : 150
  Significant co-abundance pairs       : 7671
  -> Run NB03 (03_ml_benchmarking.ipynb) next.
